# Woodbury Identity Playground

Chapter 10: Matrix Block Partitioning · *Mathematics for AI and Machine Learning* by Fuheng Wu

Compare direct inverse update with Woodbury low-rank formula.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-10/woodbury/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(0)
torch.set_printoptions(precision=5)

# The demo's default: A = diag(a, a+0.8), rank-one update U C V^T
a, u, v, c = 1.8, 0.9, -0.6, 1.2
A = torch.diag(torch.tensor([a, a + 0.8], dtype=torch.float64))
U = torch.tensor([[u], [1.0]], dtype=torch.float64)
V = torch.tensor([[v], [-0.4]], dtype=torch.float64)
C = torch.tensor([[c]], dtype=torch.float64)


def woodbury_inverse(Ainv, U, C, V):
    """(A + U C V^T)^-1 = A^-1 - A^-1 U (C^-1 + V^T A^-1 U)^-1 V^T A^-1."""
    middle = torch.linalg.inv(torch.linalg.inv(C) + V.T @ Ainv @ U)
    return Ainv - Ainv @ U @ middle @ V.T @ Ainv


direct = torch.linalg.inv(A + U @ C @ V.T)
wb = woodbury_inverse(torch.linalg.inv(A), U, C, V)
print("direct inverse =\n", direct)
print("Woodbury       =\n", wb)
print("difference ||.||_F =", torch.linalg.matrix_norm(direct - wb).item())

# ── Why it matters: a rank-k update of a big, easy-to-invert A ──────────────
n, k = 2000, 5
d = torch.rand(n, dtype=torch.float64) + 1.0
Ainv_big = torch.diag(1 / d)                 # diagonal: inverse is free
U_big = torch.randn(n, k, dtype=torch.float64)
V_big = torch.randn(n, k, dtype=torch.float64)
C_big = torch.eye(k, dtype=torch.float64)
rhs = torch.randn(n, dtype=torch.float64)
# Apply the inverse to a vector with only k x k solves
Ainv_rhs = rhs / d
small = torch.linalg.inv(C_big) + V_big.T @ (U_big / d[:, None])
x = Ainv_rhs - (U_big / d[:, None]) @ torch.linalg.solve(small, V_big.T @ Ainv_rhs)
M = torch.diag(d) + U_big @ C_big @ V_big.T
print(f"\nn={n}, rank-{k} update: residual of the Woodbury solve =", (M @ x - rhs).norm().item())

direct inverse =
 tensor([[0.99474, 0.20270],
        [0.33784, 0.54054]], dtype=torch.float64)
Woodbury       =
 tensor([[0.99474, 0.20270],
        [0.33784, 0.54054]], dtype=torch.float64)
difference ||.||_F = 2.7755575615628914e-17

n=2000, rank-5 update: residual of the Woodbury solve = 6.823444424454329e-12
